In [ ]:
pip install xgboost

In [ ]:
import mlflow
import mlflow.xgboost
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

experiment = mlflow.set_experiment("/xgboost-hipervalorizacion")

with mlflow.start_run(experiment_id=experiment.experiment_id, run_name="XGBoost_Log_IQR"):

    print("--- 1. CARGA Y PREPARACIÓN DE DATOS ---")
    matriz = "/Volumes/workspace/default/datos_bogota/matriz_barrios_unificada.csv"
    df = pd.read_csv(matriz)

    # Filtrar registros sin variable objetivo
    df_clean = df.dropna(subset=['v_ref_2026']).copy()

    # Transformaciones logarítmicas
    df_clean['log_v_ref_2026'] = np.log(df_clean['v_ref_2026'])
    df_clean['log_densidad_airbnb'] = np.log1p(df_clean['densidad_airbnb_km2'])

    # Filtro de atípicos (IQR sobre log del precio)
    Q1 = df_clean['log_v_ref_2026'].quantile(0.25)
    Q3 = df_clean['log_v_ref_2026'].quantile(0.75)
    IQR = Q3 - Q1
    df_model = df_clean[(df_clean['log_v_ref_2026'] >= Q1 - 1.5 * IQR) &
                        (df_clean['log_v_ref_2026'] <= Q3 + 1.5 * IQR)].copy()
    print(f"Registros sin NaNs: {len(df_clean)} | Tras filtrar atípicos: {len(df_model)}")

    variables_predictoras = [
        'log_densidad_airbnb',
        'estrato_predominante',
        'num_estaciones_transmilenio',
        'cagr_15_25'
    ]
    X = df_model[variables_predictoras]
    y = df_model['log_v_ref_2026']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Registrar info del dataset
    mlflow.log_param("n_registros_modelo", len(df_model))
    mlflow.log_param("variables", ", ".join(variables_predictoras))

    print("--- 2. ENTRENAMIENTO ---")
    params = {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "objective": "reg:squarederror"
    }
    mlflow.log_params(params)

    modelo_xgb = xgb.XGBRegressor(**params)
    modelo_xgb.fit(X_train, y_train)

    print("--- 3. EVALUACIÓN ---")
    y_pred_log = modelo_xgb.predict(X_test)
    y_pred_real = np.exp(y_pred_log)
    y_test_real = np.exp(y_test)

    r2_xgb = r2_score(y_test, y_pred_log)
    mae_xgb = mean_absolute_error(y_test_real, y_pred_real)
    mse_xgb = mean_squared_error(y_test_real, y_pred_real)
    rmse_xgb = np.sqrt(mse_xgb)

    mlflow.log_metric("r2", r2_xgb)
    mlflow.log_metric("mae", mae_xgb)
    mlflow.log_metric("mse", mse_xgb)
    mlflow.log_metric("rmse", rmse_xgb)

    print(f"R2: {r2_xgb:.4f} | MAE: ${mae_xgb:,.0f} COP/m² | RMSE: ${rmse_xgb:,.0f} COP/m²")

    # Gráfico de importancia de variables como artefacto
    importancias = modelo_xgb.feature_importances_
    indices = np.argsort(importancias)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(len(indices)), importancias[indices], color='teal')
    ax.set_yticks(range(len(indices)))
    ax.set_yticklabels([variables_predictoras[i] for i in indices])
    ax.set_title("Importancia de las Variables - XGBoost")
    ax.set_xlabel("Importancia Relativa")
    plt.tight_layout()
    mlflow.log_figure(fig, "importancia_variables.png")
    plt.show()

    # Registrar el modelo
    mlflow.xgboost.log_model(modelo_xgb, "xgboost_model", input_example=X_train.head(5))

    print("¡Experimento XGBoost registrado en MLflow!")